# A guide Portfolio Optimization Environment

This notebook aims to provide an example of using PortfolioOptimizationEnv (or POE) to train a reinforcement learning model that learns to solve the portfolio optimization problem.

In this document, we will reproduce a famous architecture called EIIE (ensemble of identical independent evaluators), introduced in the following paper:

- Zhengyao Jiang, Dixing Xu, & Jinjun Liang. (2017). A Deep Reinforcement Learning Framework for the Financial Portfolio Management Problem. https://doi.org/10.48550/arXiv.1706.10059.

It's advisable to read it to understand the algorithm implemented in this notebook.

### Note
If you're using this environment, consider citing the following paper (in adittion to FinRL references):

- Caio Costa, & Anna Costa (2023). POE: A General Portfolio Optimization Environment for FinRL. In *Anais do II Brazilian Workshop on Artificial Intelligence in Finance* (pp. 132–143). SBC. https://doi.org/10.5753/bwaif.2023.231144.

```
@inproceedings{bwaif,
 author = {Caio Costa and Anna Costa},
 title = {POE: A General Portfolio Optimization Environment for FinRL},
 booktitle = {Anais do II Brazilian Workshop on Artificial Intelligence in Finance},
 location = {João Pessoa/PB},
 year = {2023},
 keywords = {},
 issn = {0000-0000},
 pages = {132--143},
 publisher = {SBC},
 address = {Porto Alegre, RS, Brasil},
 doi = {10.5753/bwaif.2023.231144},
 url = {https://sol.sbc.org.br/index.php/bwaif/article/view/24959}
}

```

## Installation and imports

To run this notebook in google colab, uncomment the cells below.

In [1]:
## install finrl library
# !sudo apt install swig
# !pip install git+https://github.com/AI4Finance-Foundation/FinRL.git

In [2]:
## We also need to install quantstats, because the environment uses it to plot graphs
# !pip install quantstats

In [3]:
## Hide matplotlib warnings
# import warnings
# warnings.filterwarnings('ignore')

import logging
logging.getLogger('matplotlib.font_manager').disabled = True

#### Import the necessary code libraries

In [4]:
import pandas as pd
import yfinance as yf
from datetime import datetime
class YahooRealtimeDownloader:
    """
    Provides methods for retrieving near-real-time (1-minute) stock data
    from Yahoo Finance API, returning only the last available bar or
    "one minute behind" the latest bar.
    """

    def __init__(self, ticker_list: list):
        """
        Parameters
        ----------
        ticker_list: list
            a list of stock tickers
        """
        self.ticker_list = ticker_list

    def fetch_data(self, proxy=None, pick_second_to_last=True) -> pd.DataFrame:
        """
        Fetches near-real-time 1-minute data from Yahoo API for the current day.
        """
        import datetime
        data_df = pd.DataFrame()
        num_failures = 0

        # 按照 1 分钟周期下载当日数据
        for tic in self.ticker_list:
            temp_df = yf.download(
                tickers=tic,
                period='1d',
                interval='1m',
                proxy=proxy,
                progress=False
            )

            temp_df["tic"] = tic

            if len(temp_df) > 0:
                # 如果您想获取倒数第二条数据
                if pick_second_to_last and len(temp_df) > 1:
                    temp_df = temp_df.iloc[[-2]]  # 取倒数第二行
                else:
                    temp_df = temp_df.iloc[[-1]]

                data_df = pd.concat([data_df, temp_df], axis=0)
            else:
                num_failures += 1

        if num_failures == len(self.ticker_list):
            raise ValueError("No data is fetched. Possibly all tickers returned empty for today.")

        # reset the index
        data_df = data_df.reset_index()

        # rename columns
        try:
            data_df.columns = [
                "date",
                "open",
                "high",
                "low",
                "close",
                "adjcp",
                "volume",
                "tic",
            ]
            data_df["close"] = data_df["adjcp"]
            data_df = data_df.drop(labels="adjcp", axis=1)
        except ValueError:
            print("Columns might not match the expected format; please check yfinance returned columns.")

        # 将日期列转换为 datetime
        data_df["date"] = pd.to_datetime(data_df["date"])
        
        # 获取本地当前日期，例如 2023-10-10
        today = datetime.date.today()

        # 如果行里的 date 不是当天，就改为当天；去掉时分秒，保留 YYYY-MM-DD
        def fix_date(dt):
            if dt.date() != today:
                return today.strftime('%Y-%m-%d')
            else:
                return dt.strftime('%Y-%m-%d')

        data_df["date"] = data_df["date"].apply(fix_date)

        # 再创建 day 列，这里仅保留日期而无时分秒，所以先转回 datetime
        data_df["date"] = pd.to_datetime(data_df["date"])
        data_df["day"] = data_df["date"].dt.dayofweek
        data_df["date"] = data_df["date"].dt.strftime("%Y-%m-%d")

        data_df = data_df.dropna().reset_index(drop=True)

        data_df = data_df.sort_values(by=["date", "tic"]).reset_index(drop=True)

        print("Shape of realtime DataFrame: ", data_df.shape)
        print(data_df)
        return data_df

In [5]:
import torch

import numpy as np

from sklearn.preprocessing import MaxAbsScaler

import sys
import os

# 获取当前文件的绝对路径，并向上追溯两级到项目根目录
project_root = os.path.dirname(os.path.abspath(os.getcwd()))
print(project_root)
sys.path.append(project_root)

from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import GroupByScaler
from finrl.meta.env_portfolio_optimization.env_portfolio_optimization import PortfolioOptimizationEnv
from finrl.agents.portfolio_optimization.models import DRLAgent
from finrl.agents.portfolio_optimization.architectures import EIIE

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

/home/stock/projects/finstock/FinRL


/mnt/data/miniconda3/envs/finstock/lib/python3.10/site-packages/torch_geometric/typing.py:86: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /mnt/data/miniconda3/envs/finstock/lib/python3.10/site-packages/torch_scatter/_version_cuda.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
/mnt/data/miniconda3/envs/finstock/lib/python3.10/site-packages/torch_geometric/typing.py:124: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: /mnt/data/miniconda3/envs/finstock/lib/python3.10/site-packages/torch_sparse/_version_cuda.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev
  warnings.warn(f"An issue occurred while importing 'torch-sparse'. "
/mnt/data/miniconda3/envs/finstock/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.

## Fetch data

In his paper, *Jiang et al* creates a portfolio composed by the top-11 cryptocurrencies based on 30-days volume. Since it's not specified when this classification was done, it's difficult to reproduce, so we will use a similar approach in the Brazillian stock market:

- We select top-10 stocks from Brazillian stock market;
- For simplicity, we disconsider stocks that have missing data for the days in period 2011-01-01 to 2019-12-31 (9 years);

In [6]:
market = "us"
if market.lower() == "us":
    TOP_BRL = [
        'BILI', 'DUO', 'NIO', 'JD', 'YINN', 'YANG', 'FUTU','XHG','BABA','PDD','RGTI'
    ]
elif market.lower() == "hk":
    TOP_BRL = [
         '1812.hk', '3900.hk', '2777.hk', '1810.hk', '2878.HK','0029.hk'
    ]
elif market.lower() == "ch":
    TOP_BRL = [
         '603063.ss', '603319.ss', '000657.sz','002640.sz','002664.sz','300182.sz','688200.SS','002850.SZ'
    ]

    # TOP_BRL = [
    # '000063.SZ',
    # '002068.SZ',
    # '002138.SZ',
    # '002779.SZ',
    # '002850.SZ',
    # '300408.SZ',
    # '300442.SZ',
    # '300657.SZ',
    # '300673.SZ',
    # '300909.SZ',
    # '300913.SZ',
    # '601111.SS',
    # '601689.SS',
    # '603063.SS',
    # '603236.SS',
    # '603305.SS',
    # '603556.SS',
    # '603667.SS',
    # '688088.SS',
    # '688160.SS',
    # '688200.SS']
else:
    raise ValueError("market 变量必须为 'us' 或 'hk'")

print("当前选择的市场:", market)
print("股票列表:", TOP_BRL)
# '9988.hk',,'1797.hk''1918.hk', '3319.hk',

当前选择的市场: us
股票列表: ['BILI', 'DUO', 'NIO', 'JD', 'YINN', 'YANG', 'FUTU', 'XHG', 'BABA', 'PDD', 'RGTI']


In [7]:
print(len(TOP_BRL))

portfolio_raw_df = YahooDownloader(start_date = '2022-01-01',
                                end_date = '2025-03-26',
                                ticker_list = TOP_BRL).fetch_data()
portfolio_raw_df

11


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

Shape of DataFrame:  (8800, 8)


Price,date,close,high,low,open,volume,tic,day
0,2022-01-03,116.256577,121.949997,115.820000,119.379997,16775300,BABA,0
1,2022-01-03,44.389999,46.349998,43.689999,46.009998,4914300,BILI,0
2,2022-01-03,98.324997,123.750000,90.000000,117.000000,17157,DUO,0
3,2022-01-03,42.060001,44.000000,40.700001,43.290001,3671300,FUTU,0
4,2022-01-03,63.830887,69.489998,66.870003,68.900002,9374400,JD,0
...,...,...,...,...,...,...,...,...
8795,2025-03-12,117.860001,119.430000,117.010002,117.769997,5632000,PDD,2
8796,2025-03-12,8.950000,9.100000,8.040000,8.370000,70570100,RGTI,2
8797,2025-03-12,0.840000,0.867000,0.800000,0.800000,66800,XHG,2
8798,2025-03-12,38.610001,39.840000,38.340000,38.500000,3712700,YANG,2


In [9]:
realtime_df = YahooRealtimeDownloader(ticker_list = TOP_BRL).fetch_data()
# 假设这两个 DataFrame 的列名相同 
portfolio_raw_df = pd.concat([portfolio_raw_df, realtime_df], ignore_index=True)

# 对合并后的数据，根据日期和股票标的排序，并重新索引
portfolio_raw_df = portfolio_raw_df.sort_values(["date", "tic"]).reset_index(drop=True)

# 检查合并后的数据
print(portfolio_raw_df.head())
print(portfolio_raw_df.tail())

Columns might not match the expected format; please check yfinance returned columns.


KeyError: 'date'

In [10]:
from finrl.config import INDICATORS
from finrl.meta.preprocessor.preprocessors import FeatureEngineer
fe = FeatureEngineer(
    use_technical_indicator=True,
    tech_indicator_list=INDICATORS,
    use_vix=False,
    use_turbulence=False,
    user_defined_feature=False
)
processed = fe.preprocess_data(portfolio_raw_df)
logging.info("数据预处理完成。")

Successfully added technical indicators


In [ ]:
# from feature.ch_feature_engineer import ChFeatureEngineer
# ch_fe = ChFeatureEngineer()
# portfolio_processed = ch_fe.preprocess_data(processed)
# logging.info("自定义特征工程完成。")

In [11]:
import datetime
import pandas as pd
# 节假日日期字典（已修正）
holidays = {
    '春节': {
        '2018': '2018-02-15',
        '2019': '2019-02-05',
        '2020': '2020-01-24',
        '2021': '2021-02-11',
        '2022': '2022-01-31',
        '2023': '2023-01-21',
        '2024': '2024-02-10',
        '2025': '2025-01-28'
    },
    # '五一': {
    #     '2018': '2018-05-01',
    #     '2019': '2019-05-01',
    #     '2020': '2020-05-01',
    #     '2021': '2021-05-01',
    #     '2022': '2022-05-01',
    #     '2023': '2023-05-01',
    #     '2024': '2024-05-01',
    #     '2025': '2025-05-01'
    # },
    # '国庆': {
    #     '2018': '2018-10-01',
    #     '2019': '2019-10-01',
    #     '2020': '2020-10-01',
    #     '2021': '2021-10-01',
    #     '2022': '2022-10-01',
    #     '2023': '2023-10-01',
    #     '2024': '2024-10-01',
    #     '2025': '2025-10-01'
    # }
}

# 只选择2019年及以后的日期
selected_holidays = {}
for holiday, years in holidays.items():
    selected_holidays[holiday] = {}
    for year, date_str in years.items():
        if int(year) >= 2019:
            selected_holidays[holiday][year] = date_str

# 生成节假日前后18天的日期范围
date_ranges = []
yesterday = datetime.date.today() - datetime.timedelta(days=0)
for holiday, years in selected_holidays.items():
    for year, date_str in years.items():
        holiday_date = datetime.datetime.strptime(date_str, '%Y-%m-%d').date()
        start_date = holiday_date - datetime.timedelta(days=0)
        end_date = holiday_date + datetime.timedelta(days=35)
        today = datetime.date.today()
        if end_date > yesterday:
            end_date = yesterday
        date_ranges.append((start_date, end_date))


# 找到所有日期范围的最小开始日期和最大结束日期
overall_start_date = min([start for start, end in date_ranges])
overall_end_date = max([end for start, end in date_ranges])

print(f"\n总体下载日期范围：{overall_start_date} 至 {overall_end_date}")



# 下载数据
# logging.info("开始下载股票数据...")
# portfolio_raw_df = YahooDownloader(
#     start_date=str(overall_start_date),
#     end_date=str(overall_end_date),
#     ticker_list=TOP_BRL
# ).fetch_data()
# logging.info("股票数据下载完成。")

# 确保 'date' 列为 datetime 类型
portfolio_raw_df['date'] = pd.to_datetime(portfolio_raw_df['date']).dt.date

# 初始化一个空的DataFrame来存储筛选后的数据
filtered_df = pd.DataFrame()

# 遍历每个日期范围，并筛选数据
for start, end in date_ranges:
    print(start,end)
    mask = (portfolio_raw_df['date'] >= start) & (portfolio_raw_df['date'] <= end)
    temp_df = portfolio_raw_df.loc[mask]
    # print(temp_df.tail())
    filtered_df = pd.concat([filtered_df, temp_df], ignore_index=True)
    print(filtered_df.tail())
filtered_df['date'] = filtered_df['date'].astype(str)
filtered_df = filtered_df.sort_values(['date', 'tic']).reset_index(drop=True)
# 删除重复的数据（如果有重叠的日期范围）
# filtered_df.drop_duplicates(subset=['date', 'tic'], inplace=True)

print("\n筛选后的数据示例：")
print(filtered_df.tail())

print(f"\n筛选后的数据总行数：{len(filtered_df)}")


总体下载日期范围：2019-02-05 至 2025-03-04
2019-02-05 2019-03-12
Empty DataFrame
Columns: [date, close, high, low, open, volume, tic, day]
Index: []
2020-01-24 2020-02-28
Empty DataFrame
Columns: [date, close, high, low, open, volume, tic, day]
Index: []
2021-02-11 2021-03-18
Empty DataFrame
Columns: [date, close, high, low, open, volume, tic, day]
Index: []
2022-01-31 2022-03-07
Price        date       close         high         low         open    volume  \
270    2022-03-07   38.610001    41.619999   37.400002    40.099998  13841600   
271    2022-03-07    6.870000     7.500000    6.690000     7.500000    496000   
272    2022-03-07  968.000000  1280.000000  888.000000  1152.000000        91   
273    2022-03-07  550.353027   574.200012  533.400024   549.799988     90475   
274    2022-03-07   97.304680   112.599998  103.400002   108.400002    525770   

Price   tic  day  
270     PDD    0  
271    RGTI    0  
272     XHG    0  
273    YANG    0  
274    YINN    0  
2023-01-21 2023-02-25
Pri

In [12]:
filtered_df.groupby("tic").count()

Price,date,close,high,low,open,volume,day
tic,,,,,,,
BABA,98,98,98,98,98,98,98
BILI,98,98,98,98,98,98,98
DUO,98,98,98,98,98,98,98
FUTU,98,98,98,98,98,98,98
JD,98,98,98,98,98,98,98
NIO,98,98,98,98,98,98,98
PDD,98,98,98,98,98,98,98
RGTI,98,98,98,98,98,98,98
XHG,98,98,98,98,98,98,98


### Normalize Data

We normalize the data dividing the time series of each stock by its maximum value, so that the dataframe contains values between 0 and 1.

In [15]:
portfolio_norm_df = GroupByScaler(by="tic", scaler=MaxAbsScaler).fit_transform(filtered_df)
portfolio_norm_df

/mnt/data/miniconda3/envs/finstock/lib/python3.10/site-packages/FinRL-0.3.7-py3.10.egg/finrl/meta/preprocessor/preprocessors.py:101: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.17119622 0.12651432 0.11821808 0.14010635 0.11391093 0.24055061
 0.18797382 0.15358648 0.13560568 0.11687015 0.11068583 0.12313692
 0.15139631 0.13383369 0.17838938 0.18304824 0.20605306 0.29697095
 0.18029937 0.19405716 0.17853326 0.17494299 0.18280255 0.20677246
 0.17399726 0.11434846 0.06568924 0.08165824 0.09399322 0.15168323
 0.18442057 0.1289872  0.13986655 0.19686241 0.13447484 0.12905031
 0.14187498 0.12056896 0.13699148 0.15798198 0.15598448 0.10920075
 0.1533896  0.18298093 0.15247583 0.22863213 0.20600931 0.34062466
 0.30437362 0.15272404 0.1140994  0.10784693 0.10162559 0.13774622
 0.12076921 0.19801429 0.12483572 0.10350697 0.11757019 0.12192109
 0.14224184 0.10558945 0.1005974  0.16449528 0.13297966 0.16647427
 

Price,date,close,high,low,open,volume,tic,day
0,2022-01-31,0.845087,0.866277,0.832743,0.834534,0.171196,BABA,0.00
1,2022-01-31,0.932611,0.905115,0.863260,0.848930,0.238126,BILI,0.00
2,2022-01-31,0.957447,0.884211,0.820896,0.783721,0.000062,DUO,0.00
3,2022-01-31,0.350032,0.330685,0.331740,0.317389,0.727838,FUTU,0.00
4,2022-01-31,0.956444,0.952714,0.937084,0.935051,0.171341,JD,0.00
...,...,...,...,...,...,...,...,...
1073,2025-03-04,0.838087,0.820078,0.813665,0.810420,0.313730,PDD,0.25
1074,2025-03-04,0.568330,0.543894,0.527007,0.501242,0.251690,RGTI,0.25
1075,2025-03-04,0.000432,0.000338,0.000409,0.000327,0.003894,XHG,0.25
1076,2025-03-04,0.079404,0.079868,0.079809,0.081484,0.761011,YANG,0.25


In [22]:
df_portfolio = portfolio_norm_df[["date", "tic", "close", "high", "low",'volume']]

df_portfolio_train = df_portfolio[(df_portfolio["date"] >= "2019-01-01") & (df_portfolio["date"] <= "2025-01-29")]

df_portfolio_2025 = df_portfolio[(df_portfolio["date"] >= "2021-09-01") & (df_portfolio["date"] <= "2025-03-13")]

unique_dates = df_portfolio_2025['date'].drop_duplicates().sort_values().reset_index(drop=True)
print(unique_dates)

0     2022-01-31
1     2022-02-01
2     2022-02-02
3     2022-02-03
4     2022-02-04
         ...    
93    2025-02-26
94    2025-02-27
95    2025-02-28
96    2025-03-03
97    2025-03-04
Name: date, Length: 98, dtype: object


### Instantiate Environment

Using the `PortfolioOptimizationEnv`, it's easy to instantiate a portfolio optimization environment for reinforcement learning agents. In the example below, we use the dataframe created before to start an environment.

In [17]:
features=["close", "high", "low",'volume']
environment = PortfolioOptimizationEnv(
        df_portfolio_train,
        initial_amount=100000,
        comission_fee_pct=0.0025,
        time_window=4,
        features=features,
        normalize_df=None
    )

### Instantiate Model

Now, we can instantiate the model using FinRL API. In this example, we are going to use the EIIE architecture introduced by Jiang et. al.

:exclamation: **Note:** Remember to set the architecture's `time_window` parameter with the same value of the environment's `time_window`.

In [18]:
# set PolicyGradient parameters
model_kwargs = {
    "lr": 0.01,
    "policy": EIIE,
}

# here, we can set EIIE's parameters
policy_kwargs = {
    "k_size": 3,
    "time_window": 4,
    "initial_features":len(features)
}

model = DRLAgent(environment).get_model("pg", device, model_kwargs, policy_kwargs)

model_name = "AFTER"
file_path = f"policy_EIIE_US_{model_name}.pt"
import os
import torch

if market.lower() == "us":
    if os.path.isfile(file_path):
        model.train_policy.load_state_dict(torch.load(file_path))
        print(f"成功加载模型参数：{file_path}")
    else:
        print(f"未找到模型文件: {file_path}，请确认路径或先进行保存。")
elif market.lower() == "hk":
    if os.path.isfile(file_path):
        model.train_policy.load_state_dict(torch.load("policy_EIIE_HK.pt"))
        print("成功加载模型参数：policy_EIIE_HK.pt")
    else:
        print(f"未找到模型文件: {file_path}，请确认路径或先进行保存。")
elif market.lower() == "ch":
    model.train_policy.load_state_dict(torch.load("policy_EIIE_CH3.pt"))
    print("成功加载模型参数：policy_EIIE_CH3.pt")
else:
    raise ValueError("market 变量必须为 'us' 或 'hk'")

成功加载模型参数：policy_EIIE_US_AFTER.pt


### Train Model

In [19]:
DRLAgent.train_model(model, episodes=40)

if market.lower() == "us":  
    torch.save(model.train_policy.state_dict(), "policy_EIIE_US_{}.pt".format(model_name))

elif market.lower() == "hk":
    torch.save(model.train_policy.state_dict(), "policy_EIIE_HK.pt")
elif market.lower() == "ch":
    torch.save(model.train_policy.state_dict(), "policy_EIIE_CH3.pt")
else:
    raise ValueError("market 变量必须为 'us' 或 'hk'")

  0%|          | 0/40 [00:00<?, ?it/s]

Initial portfolio value:100000
Final portfolio value: 108129.75
Final accumulative portfolio value: 1.081297516822815
Maximum DrawDown: -0.1701009785299974
Sharpe ratio: 0.8774423983374857


  2%|▎         | 1/40 [00:01<00:49,  1.26s/it]

Initial portfolio value:100000
Final portfolio value: 125248.171875
Final accumulative portfolio value: 1.2524816989898682
Maximum DrawDown: -0.1552182281145721
Sharpe ratio: 1.9420807547930818


  5%|▌         | 2/40 [00:02<00:47,  1.25s/it]

...

 95%|█████████▌| 38/40 [00:48<00:02,  1.26s/it]

Initial portfolio value:100000
Final portfolio value: 322254.3125
Final accumulative portfolio value: 3.222543239593506
Maximum DrawDown: -0.1551489510792592
Sharpe ratio: 2.839808795174138


 98%|█████████▊| 39/40 [00:49<00:01,  1.25s/it]

Initial portfolio value:100000
Final portfolio value: 324044.125
Final accumulative portfolio value: 3.24044132232666
Maximum DrawDown: -0.15499872123089775
Sharpe ratio: 2.847596713858746


100%|██████████| 40/40 [00:50<00:00,  1.27s/it]


### Save Model

## Test Model

### Instantiate different environments

Since we have three different periods of time, we need three different environments instantiated to simulate them.

In [20]:
policy = EIIE(time_window=4, device=device,initial_features=len(features))
policy.load_state_dict(torch.load("policy_EIIE_CH3.pt"))
if market.lower() == "us":
    policy.load_state_dict(torch.load("policy_EIIE_US.pt"))
elif market.lower() == "hk":
    policy.load_state_dict(torch.load("policy_EIIE_HK.pt"))
elif market.lower() == "ch":
    policy.load_state_dict(torch.load("policy_EIIE_CH3.pt"))
else:
    raise ValueError("market 变量必须为 'us' 或 'hk'")

environment_2025 = PortfolioOptimizationEnv(
    df_portfolio_2025,
    initial_amount=100000,
    comission_fee_pct=0.0025,
    time_window=4,
    features=features,
    normalize_df=None
)
# df_account_value_ppo, df_actions_ppo = DRLAgent.DRL_prediction(
#     model=model, 
#     environment = environment_2025)
DRLAgent.DRL_validation(model, environment_2025, policy=policy)
# environment_2021 = PortfolioOptimizationEnv(
#     df_portfolio_2021,
#     initial_amount=100000,
#     comission_fee_pct=0.0025,
#     time_window=50,
#     features=["close", "high", "low"],
#     normalize_df=None
# )

# environment_2022 = PortfolioOptimizationEnv(
#     df_portfolio_2022,
#     initial_amount=100000,
#     comission_fee_pct=0.0025,
#     time_window=50,
#     features=["close", "high", "low"],
#     normalize_df=None
# )

# for i, action in enumerate(environment_2025._actions_memory):
#     if not np.isnan(action).all():  # 如果 action 中不全是 NaN
#         if i < len(unique_dates):
#             current_date = unique_dates.iloc[i]
#         else:
#             current_date = '未知日期'  # 处理索引超出范围的情况
#         print(f"Action on {current_date} at step {i}: {action}")

# columns = ["date", "cash"] + TOP_BRL  
# results = []
unique_dates = df_portfolio_2025['date'].drop_duplicates().sort_values().reset_index(drop=True)

columns = ["date", "cash"] + TOP_BRL  
results = []

shift_value = 3  # 想统一日期往后移 4 格
actions = environment_2025._actions_memory

for i, action in enumerate(actions):
    # 如果 action 全是 NaN，我们跳过
    if np.isnan(action).all():
        continue
    
    shifted_index = i + shift_value
    if shifted_index < len(unique_dates):
        current_date = unique_dates.iloc[shifted_index]
    else:
        current_date = '未知日期'
    
    # 将 action 转成百分比字符串，比如 0.123 -> "12.30%"
    action_in_percent = [f"{x*100:.2f}%" for x in action]
    
    # 构建一行 [日期, 第一列现金比例, 后面的列是各股票比例]
    row = [current_date] + action_in_percent
    results.append(row)

# 创建 DataFrame
df_action_percent = pd.DataFrame(results, columns=columns)
print("日期统一往后移 3 格后的 DataFrame：")
print(df_action_percent.tail(10))



Initial portfolio value:100000
Final portfolio value: 127106.6953125
Final accumulative portfolio value: 1.2710669040679932
Maximum DrawDown: -0.5786823228443063
Sharpe ratio: 1.0863618759517144
日期统一往后移 3 格后的 DataFrame：
          date   cash   BILI    DUO    NIO     JD   YINN     YANG   FUTU  \
85  2025-02-19  0.00%  0.00%  0.00%  0.00%  0.00%  0.00%    0.00%  0.00%   
86  2025-02-20  0.00%  0.00%  0.00%  0.00%  0.00%  0.00%    0.00%  0.00%   
87  2025-02-21  0.00%  0.00%  0.11%  0.00%  0.00%  0.00%    0.00%  0.00%   
88  2025-02-24  0.00%  0.00%  0.00%  0.00%  0.00%  0.00%    0.00%  0.00%   
89  2025-02-25  0.00%  0.00%  0.00%  0.00%  0.00%  0.00%    0.00%  0.00%   
90  2025-02-26  0.00%  0.00%  0.00%  0.00%  0.00%  0.00%    0.00%  0.00%   
91  2025-02-27  0.00%  0.00%  0.00%  0.00%  0.00%  0.00%  100.00%  0.00%   
92  2025-02-28  0.00%  0.00%  0.00%  0.00%  0.00%  0.00%  100.00%  0.00%   
93  2025-03-03  0.00%  0.00%  0.00%  0.00%  0.00%  0.00%  100.00%  0.00%   
94  2025-03-04  0.00

In [ ]:
# unique_dates = df_portfolio['date'].drop_duplicates().sort_values().reset_index(drop=True)


### Test EIIE architecture
Now, we can test the EIIE architecture in the three different test periods. It's important no note that, in this code, we load the saved policy even though it's not necessary just to show how to save and load your model.

In [21]:
EIIE_results = {
    "training": environment._asset_memory["final"],
    "2025": {},

}

# instantiate an architecture with the same arguments used in training
# and load with load_state_dict.
policy = EIIE(time_window=50, device=device)
policy.load_state_dict(torch.load("policy_EIIE_US.pt"))

# 2020
DRLAgent.DRL_validation(model, environment_2025, policy=policy)
EIIE_results["2025"]["value"] = environment_2025._asset_memory["final"]

# # 2021
# DRLAgent.DRL_validation(model, environment_2021, policy=policy)
# EIIE_results["2021"]["value"] = environment_2021._asset_memory["final"]

# # 2022
# DRLAgent.DRL_validation(model, environment_2022, policy=policy)
# EIIE_results["2022"]["value"] = environment_2022._asset_memory["final"]

RuntimeError: Error(s) in loading state_dict for EIIE:
	size mismatch for sequential.0.weight: copying a param with shape torch.Size([2, 4, 1, 3]) from checkpoint, the shape in current model is torch.Size([2, 3, 1, 3]).
	size mismatch for sequential.2.weight: copying a param with shape torch.Size([20, 2, 1, 2]) from checkpoint, the shape in current model is torch.Size([20, 2, 1, 48]).

In [ ]:
# unique_dates = df_portfolio['date'].drop_duplicates().sort_values().reset_index(drop=True)
unique_dates = df_portfolio_2025['date'].drop_duplicates().sort_values().reset_index(drop=True)
for i, action in enumerate(environment_2025._actions_memory):
    if not np.isnan(action).all():  # 如果 action 中不全是 NaN
        if i < len(unique_dates):
            current_date = unique_dates.iloc[i]
        else:
            current_date = '未知日期'  # 处理索引超出范围的情况
        print(f"Action on {current_date} at step {i}: {action}")

In [ ]:
filtered_df.tail()

### Test Uniform Buy and Hold
For comparison, we will also test the performance of a uniform buy and hold strategy. In this strategy, the portfolio has no remaining cash and the same percentage of money is allocated in each asset.

In [ ]:
UBAH_results = {
    "train": {},
    "2025": {},
    # "2021": {},
    # "2022": {}
}

PORTFOLIO_SIZE = len(TOP_BRL)

# train period
terminated = False
environment.reset()
while not terminated:
    action = [0] + [1/PORTFOLIO_SIZE] * PORTFOLIO_SIZE
    _, _, terminated, _ = environment.step(action)
UBAH_results["train"]["value"] = environment._asset_memory["final"]

# 2020
terminated = False
environment_2025.reset()
while not terminated:
    action = [0] + [1/PORTFOLIO_SIZE] * PORTFOLIO_SIZE
    _, _, terminated, _ = environment_2025.step(action)
UBAH_results["2025"]["value"] = environment_2025._asset_memory["final"]

# # 2021
# terminated = False
# environment_2021.reset()
# while not terminated:
#     action = [0] + [1/PORTFOLIO_SIZE] * PORTFOLIO_SIZE
#     _, _, terminated, _ = environment_2021.step(action)
# UBAH_results["2021"]["value"] = environment_2021._asset_memory["final"]

# # 2022
# terminated = False
# environment_2022.reset()
# while not terminated:
#     action = [0] + [1/PORTFOLIO_SIZE] * PORTFOLIO_SIZE
#     _, _, terminated, _ = environment_2022.step(action)
# UBAH_results["2022"]["value"] = environment_2022._asset_memory["final"]

### Plot graphics

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline 

plt.plot(UBAH_results["train"]["value"], label="Buy and Hold")
plt.plot(EIIE_results["training"], label="EIIE")

plt.xlabel("Days")
plt.ylabel("Portfolio Value")
plt.title("Performance in training period")
plt.legend()

plt.show()

In [ ]:
plt.plot(UBAH_results["2025"]["value"], label="Buy and Hold")
plt.plot(EIIE_results["2025"]["value"], label="EIIE")

plt.xlabel("Days")
plt.ylabel("Portfolio Value")
plt.title("Performance in 2025")
plt.legend()

plt.show()

In [ ]:
plt.plot(UBAH_results["2021"]["value"], label="Buy and Hold")
plt.plot(EIIE_results["2021"]["value"], label="EIIE")

plt.xlabel("Days")
plt.ylabel("Portfolio Value")
plt.title("Performance in 2021")
plt.legend()

plt.show()

In [ ]:
plt.plot(UBAH_results["2022"]["value"], label="Buy and Hold")
plt.plot(EIIE_results["2022"]["value"], label="EIIE")

plt.xlabel("Days")
plt.ylabel("Portfolio Value")
plt.title("Performance in 2022")
plt.legend()

plt.show()

We can see that the agent is able to learn a good policy but its performance is worse the more the test period advances into the future. To get a better performance in 2022, for example, the agent should probably be trained again using more recent data.